[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/NSA-hyperspectral-course-2026/blob/main/exercises/ex_06_water_quality.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🌊 Exercises: Water quality & inland/coastal water intelligence
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Build a water-quality analysis the careful way — masks first, then indices, then a critical look at what else could cause the signal. Your test sites are the clear, shallow Gulf waters of the Al Dhafra coast (UAE) and the turbid mangrove estuary of Jiquilisco Bay (El Salvador).

These exercises accompany the theme notebook [`06_ecosystem_health_water_qualityy.ipynb`](../../notebooks/06_ecosystem_health_water_qualityy.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Checking quality masks (false cloud flags over bright sabkha)
- Choosing a water mask (NDWI vs MNDWI) for the task at hand
- Computing the Floating Algae Index and recognising its false positives
- Comparing water bodies with water-masked NDCI and turbidity proxies
- Recognising bottom reflectance in optically shallow water
- What a calibrated water-quality service needs

**Data**
- `20250511_074311_00_4001` — Al Dhafra coast, Abu Dhabi, UAE (11 May 2025)
- `20250223_165546_32_4001` — Jiquilisco Bay, El Salvador (23 Feb 2025), for comparison

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/NSA-hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
UAE_ID = "20250511_074311_00_4001"   # Al Dhafra coast, Abu Dhabi, UAE — the first scene of notebook 06
ES_ID = "20250223_165546_32_4001"    # Jiquilisco Bay estuary, El Salvador — for comparison

uae = hu.get_scene(UAE_ID)
print(uae.summary())

# Read the bands WITHOUT the cloud mask (cloud=False) — Exercise 1 shows why.
valid = uae.valid_mask(cloud=False)
B = {w: uae.read_band(w, cloud=False) for w in (443, 490, 560, 645, 665, 708, 859, 1240, 1650)}
NDWI = hu.nd(B[560], B[859])
water = valid & (NDWI > 0)          # the water mask used throughout
PIXEL_KM2 = 0.0009

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(uae.rgb(mask=True)); ax.set_title("Al Dhafra coast — true colour"); ax.axis("off")
plt.show()

## Exercise 1 — The '40% cloud' scene without a cloud in the sky  ⭐

`scene.summary()` reports that about 40% of this scene is flagged as cloud, yet the true-colour image shows a
clear sky.

1. Overlay the `beta_cloud_mask` on the true-colour image.
2. What fraction of the flagged pixels lies on land (NDWI ≤ 0) and what fraction on water? What share of all
   land pixels is flagged?
3. What do you conclude about using the default `valid_mask()` for this scene, and what does that mean for any
   automatic "cloud < 10%" scene filter in a product?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`uae.masks()["cloud"]` is the boolean cloud flag. Land = `valid & (NDWI <= 0)`, water = `valid & (NDWI > 0)`.
The catalogue value is in `hu.load_catalog_index()` (column `cloud_percent`).

</details>

## Exercise 2 — Water mask first: NDWI or MNDWI?  ⭐

Every water-quality analysis starts with a water mask.

1. Compute the water area (km²) with NDWI = (G − NIR) / (G + NIR) > 0 and with MNDWI = (G − SWIR₁₆₅₀) /
   (G + SWIR₁₆₅₀) > 0.
2. Map the pixels where the two masks disagree. Where are they?
3. Which mask would you use here, and why?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

The disagreement map is `ndwi_mask != mndwi_mask` (restricted to valid pixels).

</details>

## Exercise 3 — Floating Algae Index (FAI)  ⭐⭐

The **Floating Algae Index** (Hu, 2009) measures how far the NIR reflectance rises above a straight baseline
drawn between red and SWIR:

$$\mathrm{FAI} = R_{859} - \left[R_{645} + (R_{1240} - R_{645})\,\frac{859 - 645}{1240 - 645}\right]$$

Floating vegetation (Sargassum, algal scums, water hyacinth) has a strong NIR peak, so FAI > 0, while clear water
stays near or below 0. The baseline makes FAI fairly robust to haze and glint.

1. Compute FAI over the water mask and map it with a stretch from its 1st to 99.5th percentile.
2. Show the true-colour image of the pixels with the highest FAI (top 0.1%). What are they?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Use `B[859]`, `B[645]`, `B[1240]`. Find the top pixels with `np.nanpercentile(FAI[water], 99.9)`, then zoom in with
`uae.rgb(window=(r - 15, r + 15, c - 15, c + 15))`.

</details>

## Exercise 4 — Chlorophyll and turbidity: Gulf vs estuary  ⭐⭐

Compute the water-masked NDCI (R₇₀₈ vs R₆₆₅) and the turbidity proxy (R₆₆₅ vs R₅₆₀) for the Al Dhafra scene and
for the El Salvador estuary scene (use NDWI > 0.1 there). Compare the distributions with histograms and report
their medians. What does the difference tell you about the two water bodies?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Open the second scene with `hu.get_scene(ES_ID)`. Plot both histograms on the same axes with
`histtype="step", density=True`.

</details>

## Exercise 5 — Shallow water: when you see the seafloor, not the water  ⭐⭐⭐

Much of the Al Dhafra coast is shallow, with bright sand and seagrass beds. Light reflected by the **bottom**
adds to the water signal.

1. Extract a north–south transect along column 300, from row 190 to row 475 (water pixels only). Plot the
   reflectance at 443, 560, 665 and 708 nm, and the NDCI and turbidity proxies along it.
2. Compare the mean spectra of "bright" water (R₅₆₀ above the 80th percentile of the water pixels) and "dark"
   water (below the 20th percentile).
3. Explain why NDCI and turbidity rise near the shores. How would you avoid misinterpreting this as algae or
   sediment?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`rows = np.arange(190, 476)`; values along the transect: `B[560][rows, 300]`. For the spectra, sample pixels from
both groups and average (`uae.read_cube` on the bounding window, as in the other exercise sets).

</details>

## Exercise 6 — Reflection — from proxy maps to a water-quality service  ⭐

A water authority wants a monthly chlorophyll-a map (mg m⁻³) of this coast. In a short markdown cell, list
what you need beyond the NDCI map: the in-situ data (what, where, when), how to match satellite and field
measurements, and which conditions you would flag as "unreliable" in the product.

In [ ]:
# optional scratch space — answer in a markdown cell

<details><summary>💡 Hint</summary>

Think of matchup windows (hours, 3×3 pixels), Secchi depth, chlorophyll-a lab samples, TSS, CDOM, glint,
adjacency, optically shallow water, and the revisit time of Tanager.

</details>

## Key takeaways

- Inspect masks before trusting them — the beta cloud mask flags bright salt flats as cloud.
- Mask water first, and keep away from shorelines for water-quality indices.
- FAI highlights anything with a NIR peak: ships, structures and mixed shoreline pixels as well as algae.
- In shallow water, the seafloor raises red and red-edge reflectance and mimics chlorophyll or sediment.
- Quantitative water quality needs in-situ matchups, calibration and quality flags.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC